In [336]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import re



In [337]:
name = 'TEST017_BIG_DADDY_5_CH_ECG_STIM_batt_disc.csv'
num_CH = 5
CH_first = 0

In [338]:
df = pd.read_csv(name, sep=',', dtype=str)
df.columns

Index(['SPI', ' Time', ' MISO', ' MOSI '], dtype='object')

In [339]:
print(df.head(5))

  SPI        Time  MISO  MOSI 
0   1  -24.9128ms  0x00   0x8E
1   2  -24.9069ms  0x00   0xF3
2   3  -24.9005ms  0x00   0x33
3   4  -24.8487ms  0x00   0x83
4   5  -24.8428ms  0x00   0xDE


In [340]:
df = df.drop(columns = ' MISO')
df = df.drop(columns = 'SPI')

print(df.head(5))

         Time MOSI 
0  -24.9128ms  0x8E
1  -24.9069ms  0xF3
2  -24.9005ms  0x33
3  -24.8487ms  0x83
4  -24.8428ms  0xDE


In [341]:
df.columns = ['Time', 'MOSI']

print(df.head(21))


          Time  MOSI
0   -24.9128ms  0x8E
1   -24.9069ms  0xF3
2   -24.9005ms  0x33
3   -24.8487ms  0x83
4   -24.8428ms  0xDE
5   -24.8365ms  0x34
6   -24.7842ms  0x80
7   -24.7783ms  0x41
8   -24.7720ms  0x30
9   -24.7207ms  0x80
10  -24.7148ms  0x3F
11  -24.7085ms  0x31
12  -24.6566ms  0x84
13  -24.6508ms  0x67
14  -24.6444ms  0x32
15  -24.5926ms  0x8F
16  -24.5867ms  0x0E
17  -24.5803ms  0x33
18  -24.5286ms  0x83
19  -24.5228ms  0xBF
20  -24.5164ms  0x34


In [342]:
df['MOSI'] = df['MOSI'].apply(lambda x: int(x, 16))

In [343]:
# Verifica cómo quedó
print(df.head(25))


          Time  MOSI
0   -24.9128ms   142
1   -24.9069ms   243
2   -24.9005ms    51
3   -24.8487ms   131
4   -24.8428ms   222
5   -24.8365ms    52
6   -24.7842ms   128
7   -24.7783ms    65
8   -24.7720ms    48
9   -24.7207ms   128
10  -24.7148ms    63
11  -24.7085ms    49
12  -24.6566ms   132
13  -24.6508ms   103
14  -24.6444ms    50
15  -24.5926ms   143
16  -24.5867ms    14
17  -24.5803ms    51
18  -24.5286ms   131
19  -24.5228ms   191
20  -24.5164ms    52
21  -24.4640ms   128
22  -24.4581ms    67
23  -24.4517ms    48
24  -24.4004ms   128


In [344]:
# Crea un diccionario para mapear los valores
mapeo = {
    48+CH_first: CH_first
}

# Comienza después de la última clave
ultima_clave = max(mapeo.keys())
ultimo_valor = max(mapeo.values())

for i in range(1, num_CH):
    nueva_clave = ultima_clave + i
    nuevo_valor = ultimo_valor + i
    mapeo[nueva_clave] = nuevo_valor

print(mapeo)

{48: 0, 49: 1, 50: 2, 51: 3, 52: 4}


In [345]:
# Aplica el mapeo y pon 'S' en el resto
df['type'] = df['MOSI'].astype(int).map(mapeo).fillna('S')
df = df.reset_index(drop=True)
print(df.head(40))


          Time  MOSI type
0   -24.9128ms   142    S
1   -24.9069ms   243    S
2   -24.9005ms    51  3.0
3   -24.8487ms   131    S
4   -24.8428ms   222    S
5   -24.8365ms    52  4.0
6   -24.7842ms   128    S
7   -24.7783ms    65    S
8   -24.7720ms    48  0.0
9   -24.7207ms   128    S
10  -24.7148ms    63    S
11  -24.7085ms    49  1.0
12  -24.6566ms   132    S
13  -24.6508ms   103    S
14  -24.6444ms    50  2.0
15  -24.5926ms   143    S
16  -24.5867ms    14    S
17  -24.5803ms    51  3.0
18  -24.5286ms   131    S
19  -24.5228ms   191    S
20  -24.5164ms    52  4.0
21  -24.4640ms   128    S
22  -24.4581ms    67    S
23  -24.4517ms    48  0.0
24  -24.4004ms   128    S
25  -24.3945ms    57    S
26  -24.3881ms    49  1.0
27  -24.3363ms   132    S
28  -24.3304ms    60    S
29  -24.3240ms    50  2.0
30  -24.2722ms   143    S
31  -24.2663ms    43    S
32  -24.2600ms    51  3.0
33  -24.2082ms   131    S
34  -24.2023ms   229    S
35  -24.1959ms    52  4.0
36  -24.1437ms   128    S
37  -24.1379

In [346]:
# Inicializamos idx como None
idx = None
# Recorremos los índices donde 'type' es distinto de 'S'
for i in df.index[df['type'] != 'S']:
    # Verificamos que haya al menos dos filas siguientes
    if (i + 2) < len(df):
        # Comprobamos que las dos siguientes filas tengan 'S'
        if (df.loc[i + 1, 'type'] == 'S') and (df.loc[i + 2, 'type'] == 'S'):
            idx = i
            break
    else:
        # Si no hay suficientes filas para verificar, no es un punto válido
        continue

# Si encontramos un índice válido, cortamos el dataframe
if idx is not None:
    df = df.loc[idx:].reset_index(drop=True)
else:
    # Si no se encontró un punto válido, el dataframe queda vacío o como prefieras manejarlo
    df = df.iloc[0:0].reset_index(drop=True)


In [347]:
print(df['type'][0])
print(df['type'][1])
print(df['type'][2])
print(df['type'][3])


3.0
S
S
4.0


In [348]:
# Diccionario para almacenar los arrays
resultados = {CH_first: []}
resultados_tiempos = {CH_first: []}



# Obtener el valor máximo actual de clave
ultima_clave = max(resultados.keys())

# Agregar N nuevas claves a ambos diccionarios
for i in range(1, num_CH):
    nueva_clave = ultima_clave + i
    resultados[nueva_clave] = []
    resultados_tiempos[nueva_clave] = []

# Iterar sobre el dataframe
for i, row in df.iterrows():
    tipo = row['type']
    tipo_tiempo = row['type']  
    if tipo in resultados:
        # Tomar las dos siguientes filas si existen
        sub_df = df.iloc[i+1:i+3]['MOSI']
        sub_df_times = df.iloc[i+1:i+3]['Time']
        # Guardar como array (puedes ajustar qué columnas guardar)
        resultados[tipo].append(sub_df.to_numpy())
        resultados_tiempos[tipo_tiempo].append(sub_df_times.to_numpy())
        




In [349]:

# Opcional: convertir las listas en arrays grandes (si quieres)
import numpy as np
for k in resultados:
    resultados[k] = np.concatenate(resultados[k])
    resultados_tiempos[k] = np.concatenate(resultados_tiempos[k])

# Ahora resultados[0], resultados[1], ... tienen los arrays deseados

In [350]:
for i in range (num_CH):
    print(len(resultados[i]))


316
322
315
318
320


In [351]:
# Creamos un nuevo diccionario con las filas pares eliminadas
resultados_filtrados = {}

for k, arr in resultados_tiempos.items():
    # Tomar los elementos en posiciones impares: 1, 3, 5, ...
    resultados_tiempos[k] = arr[1::2]


In [352]:
arr0 = resultados[0].flatten()

new_arr0 = []
new_arr1 = []
new_arr2 = []
new_arr3 = []
new_arr4 = []
for i in range(0, len(arr0)-1, 2):
    combined = arr0[i] * 256 + arr0[i+1]
    new_arr0.append(combined)
new_arr0 = np.array(new_arr0)



In [353]:
arr1 = resultados[1].flatten()
for i in range(0, len(arr1)-1, 2):
    combined = arr1[i] * 256 + arr1[i+1]
    new_arr1.append(combined)
new_arr1 = np.array(new_arr1)


In [354]:

arr2 = resultados[2].flatten()
for i in range(0, len(arr2)-1, 2):
    combined = arr2[i] * 256 + arr2[i+1]
    new_arr2.append(combined)
new_arr2 = np.array(new_arr2)

In [355]:

arr3 = resultados[3].flatten()
for i in range(0, len(arr3)-1, 2):
    combined = arr3[i] * 256 + arr3[i+1]
    new_arr3.append(combined)
new_arr3 = np.array(new_arr3)

In [356]:

arr4 = resultados[4].flatten()
for i in range(0, len(arr4)-1, 2):
    combined = arr4[i] * 256 + arr4[i+1]
    new_arr4.append(combined)
new_arr4 = np.array(new_arr4)

In [357]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(y=new_arr0, mode='lines', name='Señal 0'))
fig.add_trace(go.Scatter(y=new_arr1, mode='lines', name='Señal 1'))
fig.add_trace(go.Scatter(y=new_arr2, mode='lines', name='Señal 2'))
fig.add_trace(go.Scatter(y=new_arr3, mode='lines', name='Señal 3'))
fig.add_trace(go.Scatter(y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
   hovermode='x unified',    yaxis=dict(range=[0, 65536])
)

fig.show()


In [358]:

fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[0], y=new_arr0, mode='lines', name='Señal 0'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
   hovermode='x unified',    yaxis=dict(range=[0, 65536])
)

fig.show()

In [359]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[1], y=new_arr1, mode='lines', name='Señal 1'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
   hovermode='x unified',    yaxis=dict(range=[0, 65536])
)

fig.show()

In [360]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[2], y=new_arr2, mode='lines', name='Señal 2'))

# Opciones de layout
fig.update_layout(
    title='ECG',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
   hovermode='x unified',    yaxis=dict(range=[0, 65536])
)

fig.show()

In [361]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[3], y=new_arr3, mode='lines', name='Señal 3'))

# Opciones de layout
fig.update_layout(
    title='Señales muestreadas superpuestas',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
   hovermode='x unified',    yaxis=dict(range=[0, 65536])
)

fig.show()

In [362]:
fig = go.Figure()


# Agregamos cada señal como un trazo
fig.add_trace(go.Scatter(x = resultados_tiempos[4], y=new_arr4, mode='lines', name='Señal 4'))

# Opciones de layout
fig.update_layout(
    title='BATT level',
    xaxis_title='Tiempo',
    yaxis_title='Valor',
  hovermode='x unified',    yaxis=dict(range=[0, 65536])
)

fig.show()

In [363]:
ceros = np.zeros(100)  
# FFT
X = np.fft.fft(np.concatenate((ceros,(new_arr1-32768))))

# Número de muestras
N = len(X)

# Frecuencias asociadas (eje x)
freqs = np.fft.fftfreq(N, 1/f)

# Magnitud (módulo)
magnitud = np.abs(X)

# Para mostrar solo la mitad positiva (frecuencias positivas)
idxs = freqs >= 0

plt.plot(freqs[idxs], magnitud[idxs])
plt.xlabel("Frecuencia (Hz)")
plt.ylabel("Magnitud")
plt.title("Espectro de la señal")
plt.show()

NameError: name 'f' is not defined

In [ ]:

fig = go.Figure()

fig.add_trace(go.Scatter(
    x = freqs[idxs],
    y=magnitud[idxs],
    mode='lines',
    name='Valores concatenados'
))

fig.update_layout(
    title='FFT',
    xaxis_title='Frecuencia',
    yaxis_title='Magnitud',
   hovermode='x unified',    yaxis=dict(range=[0, 65536])
)

fig.show()

NameError: name 'freqs' is not defined